# TSSCI: Time-Series Super Classifier Images
## Complete Pipeline for Human Pose Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edialagerlov1/TSSCI-Pipeline/blob/main/TSSCI_Complete_Pipeline.ipynb)

**Author:** Edia Lagerlov  
**Date:** March 14, 2026  
**Based on:** Research by Yoram Segal ([Paper](https://www.preprints.org/manuscript/202304.1268/v1))

---

### Overview

This notebook implements a complete pipeline to:
1. Extract human skeleton poses from video using **MediaPipe**
2. Transform skeleton data to **TSSCI format**
3. Generate **TSSCI images** for CNN processing
4. Visualize with **dual-view interactive player**

### What is TSSCI?

TSSCI (Time-Series Super Classifier Images) converts temporal skeleton sequences into 2D images that can be processed by CNNs like EfficientNet. This enables:
- Exercise classification for remote physiotherapy
- Compact representation of movement patterns
- Use of pre-trained vision models


## 1. Setup and Installation

First, install required packages and download the MediaPipe pose model.

In [ ]:
# Install required packages
!pip install -q mediapipe opencv-python numpy matplotlib pillow

# Download MediaPipe pose model
!wget -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

print("✓ Installation complete!")

In [ ]:
# Import libraries
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.widgets import Button, Slider
from pathlib import Path
from IPython.display import HTML, Image, display
from google.colab import files
import json

print(f"MediaPipe version: {mp.__version__}")
print(f"NumPy version: {np.__version__}")
print("\n✓ Libraries imported successfully!")

## 2. Upload Your Video

Upload a video file containing human movement (exercises, running, etc.)

In [ ]:
# Upload video file
print("Please upload your video file...")
uploaded = files.upload()

# Get the uploaded filename
video_filename = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {video_filename}")

# Create output directory
!mkdir -p tssci_output
print("✓ Output directory created")

## 3. Extract Skeleton using MediaPipe

Use MediaPipe Pose to detect 33 body landmarks in each frame.

In [ ]:
def extract_skeleton_from_video(video_path, model_path='pose_landmarker_heavy.task'):
    """
    Extract skeleton keypoints from video using MediaPipe
    
    Returns:
        skeleton_data: numpy array of shape (num_frames, 33, 3)
                      where 3 = [x, y, visibility]
    """
    print(f"Processing video: {video_path}")
    
    # Create PoseLandmarker
    base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    skeleton_data = []
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break
            
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Create MediaPipe Image
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            
            # Calculate timestamp
            timestamp_ms = int(frame_count * 1000 / fps)
            
            # Detect pose
            results = landmarker.detect_for_video(mp_image, timestamp_ms)
            
            if results.pose_landmarks and len(results.pose_landmarks) > 0:
                keypoints = []
                for landmark in results.pose_landmarks[0]:
                    keypoints.append([landmark.x, landmark.y, landmark.visibility])
                skeleton_data.append(keypoints)
            
            frame_count += 1
            if frame_count % 30 == 0:
                print(f"Processed {frame_count} frames, detected {len(skeleton_data)} poses...")
    
    cap.release()
    
    print(f"\n✓ Total frames processed: {frame_count}")
    print(f"✓ Frames with detected poses: {len(skeleton_data)}")
    
    return np.array(skeleton_data), fps

# Extract skeleton
skeleton_data, video_fps = extract_skeleton_from_video(video_filename)
print(f"\nSkeleton data shape: {skeleton_data.shape}")
print(f"Video FPS: {video_fps}")

## 4. TSSCI Transformation

Convert MediaPipe skeleton to TSSCI format with special rearrangement pattern.

In [ ]:
def mediapipe_to_openpose(skeleton_data):
    """Convert MediaPipe 33 landmarks to OpenPose 25 keypoints"""
    openpose_data = []
    
    # Mapping from MediaPipe to OpenPose indices
    mp_to_op = {
        0: 0,   # Nose
        12: 2,  # Right shoulder
        14: 3,  # Right elbow
        16: 4,  # Right wrist
        11: 5,  # Left shoulder
        13: 6,  # Left elbow
        15: 7,  # Left wrist
        24: 9,  # Right hip
        26: 10, # Right knee
        28: 11, # Right ankle
        23: 12, # Left hip
        25: 13, # Left knee
        27: 14, # Left ankle
        5: 15,  # Right eye
        2: 16,  # Left eye
        8: 17,  # Right ear
        7: 18,  # Left ear
        32: 19, # Left big toe
        30: 20, # Left small toe
        29: 21, # Left heel
        31: 22, # Right big toe
        28: 24, # Right heel
    }
    
    for frame in skeleton_data:
        op_frame = np.zeros((25, 3))
        
        for mp_idx, op_idx in mp_to_op.items():
            if mp_idx < len(frame):
                op_frame[op_idx] = frame[mp_idx]
        
        # Calculate neck (midpoint between shoulders)
        if 11 < len(frame) and 12 < len(frame):
            op_frame[1] = (frame[11] + frame[12]) / 2
        
        # Calculate mid-hip
        if 23 < len(frame) and 24 < len(frame):
            op_frame[8] = (frame[23] + frame[24]) / 2
        
        openpose_data.append(op_frame)
    
    return np.array(openpose_data)

def genTSSI(data_array):
    """Generate TSSCI by rearranging skeleton keypoints"""
    pattern = [1,2,3,4,3,2,1,0,15,17,15,0,16,18,16,0,1,5,6,7,6,5,1,8,
               12,13,14,19,20,19,14,21,14,13,12,8,9,10,11,24,11,22,23,22,11,10,9,8,1]
    return data_array[:, pattern]

def normalizeTSSI(tssiTensor):
    """Normalize values to [0, 1] range"""
    all_x = tssiTensor[:,:,0].copy()
    all_y = tssiTensor[:,:,1].copy()
    
    # Normalize X
    max_x = all_x.max()
    all_x[all_x == 0] = max_x + 1
    min_x = all_x.min()
    all_x[all_x == max_x + 1] = min_x
    
    # Normalize Y
    max_y = all_y.max()
    all_y[all_y == 0] = max_y + 1
    min_y = all_y.min()
    all_y[all_y == max_y + 1] = min_y
    
    res = tssiTensor.copy()
    res[:,:,0] = (all_x - min_x) / (max_x - min_x + 1e-8)
    res[:,:,1] = (all_y - min_y) / (max_y - min_y + 1e-8)
    
    return res

def addressZeroVal(tssiVec):
    """Handle missing/zero values"""
    tssiVec = tssiVec.copy()
    for i in range(tssiVec.shape[0]):
        for j in range(tssiVec.shape[1]):
            if tssiVec[i][j][0] * tssiVec[i][j][1] * tssiVec[i][j][2] == 0:
                if j > 0:
                    tssiVec[i][j] = tssiVec[i][j-1]
    return tssiVec

def genImageByRegion(arrNpy, n_frames=49):
    """Sample frames for TSSCI image"""
    arrLen = arrNpy.shape[0]
    if arrLen < n_frames:
        indices = np.linspace(0, arrLen-1, n_frames, dtype=int)
        return arrNpy[indices]
    
    n_regions = arrLen / n_frames
    rows = []
    for i in range(n_frames):
        r = int(np.random.randint(i*n_regions, (i+1)*n_regions))
        rows.append(r)
    return arrNpy[rows]

# Apply transformations
print("[1/4] Converting to OpenPose format...")
openpose_data = mediapipe_to_openpose(skeleton_data)

print("[2/4] Applying TSSCI transformation...")
tssci_data = genTSSI(openpose_data)

print("[3/4] Normalizing data...")
tssci_data = normalizeTSSI(tssci_data)
tssci_data = addressZeroVal(tssci_data)

print("[4/4] Generating TSSCI image...")
tssci_sampled = genImageByRegion(tssci_data, n_frames=49)
tssci_image = np.zeros((49, 49, 2))
for i in range(49):
    for j in range(49):
        tssci_image[i, j, 0] = tssci_sampled[i, j, 0]  # X
        tssci_image[i, j, 1] = tssci_sampled[i, j, 1]  # Y

print(f"\n✓ TSSCI data shape: {tssci_data.shape}")
print(f"✓ TSSCI image shape: {tssci_image.shape}")

## 5. Visualize TSSCI Image

Display the TSSCI image showing X and Y movement channels.

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# X channel
axes[0].imshow(tssci_image[:,:,0], cmap='viridis')
axes[0].set_title('TSSCI - X Channel (Horizontal Movement)', fontsize=12)
axes[0].axis('off')

# Y channel
axes[1].imshow(tssci_image[:,:,1], cmap='plasma')
axes[1].set_title('TSSCI - Y Channel (Vertical Movement)', fontsize=12)
axes[1].axis('off')

# Combined
combined = np.zeros((tssci_image.shape[0], tssci_image.shape[1], 3))
combined[:,:,0] = tssci_image[:,:,0]
combined[:,:,1] = tssci_image[:,:,1]
combined[:,:,2] = (tssci_image[:,:,0] + tssci_image[:,:,1]) / 2
axes[2].imshow(combined)
axes[2].set_title('TSSCI - Combined RGB', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('tssci_output/tssci_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ TSSCI visualization saved!")

## 6. Create Skeleton Animation

Generate animated GIF showing the skeleton running.

In [ ]:
def create_skeleton_animation(tssci_data, output_path, fps=15, duration=5):
    """Create animated GIF of skeleton"""
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)
    ax.set_title('TSSCI Skeleton Animation', fontsize=14)
    
    num_frames = min(tssci_data.shape[0], int(duration * fps))
    
    # Create connections (sequential)
    connections = [(i, i+1) for i in range(48)]
    
    lines = []
    for _ in connections:
        line, = ax.plot([], [], 'b-', linewidth=2)
        lines.append(line)
    
    points, = ax.plot([], [], 'ro', markersize=6)
    frame_text = ax.text(0.02, 0.98, '', transform=ax.transAxes,
                        fontsize=12, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    def init():
        for line in lines:
            line.set_data([], [])
        points.set_data([], [])
        return lines + [points]
    
    def animate(frame_idx):
        frame_data = tssci_data[frame_idx]
        x_coords = frame_data[:, 0]
        y_coords = frame_data[:, 1]
        
        for i, (p1, p2) in enumerate(connections):
            if p1 < len(x_coords) and p2 < len(x_coords):
                lines[i].set_data([x_coords[p1], x_coords[p2]],
                                 [y_coords[p1], y_coords[p2]])
        
        points.set_data(x_coords, y_coords)
        frame_text.set_text(f'Frame: {frame_idx + 1}/{num_frames}')
        
        return lines + [points, frame_text]
    
    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                  frames=num_frames, interval=1000/fps,
                                  blit=True)
    
    anim.save(output_path, writer='pillow', fps=fps)
    plt.close()
    print(f"✓ Animation saved to: {output_path}")

# Create animation
print("Creating skeleton animation (this may take a moment)...")
create_skeleton_animation(tssci_data, 'tssci_output/skeleton_animation.gif', 
                         fps=15, duration=5)

# Display the animation
from IPython.display import Image as IPImage
display(IPImage('tssci_output/skeleton_animation.gif'))

## 7. Create Dual-View Animation

Show TSSCI image with scanning line alongside skeleton animation.

In [ ]:
def create_dual_view_animation(tssci_image, tssci_data, output_path, fps=15, duration=5):
    """Create dual-view animation: TSSCI image + skeleton"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Setup TSSCI image
    tssci_rgb = np.zeros((tssci_image.shape[0], tssci_image.shape[1], 3))
    tssci_rgb[:,:,0] = tssci_image[:,:,0]
    tssci_rgb[:,:,1] = tssci_image[:,:,1]
    tssci_rgb[:,:,2] = (tssci_image[:,:,0] + tssci_image[:,:,1]) / 2
    
    ax1.imshow(tssci_rgb, aspect='auto')
    ax1.set_title('TSSCI Image', fontsize=12)
    ax1.set_xlabel('Keypoint Index (49 points)')
    ax1.set_ylabel('Time Frame (49 samples)')
    scan_line = ax1.axhline(y=0, color='yellow', linewidth=2, linestyle='--')
    
    # Setup skeleton
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.set_aspect('equal')
    ax2.invert_yaxis()
    ax2.grid(True, alpha=0.3)
    ax2.set_title('Skeleton Animation', fontsize=12)
    
    connections = [(i, i+1) for i in range(48)]
    lines = []
    for _ in connections:
        line, = ax2.plot([], [], 'b-', linewidth=2)
        lines.append(line)
    points, = ax2.plot([], [], 'ro', markersize=6)
    
    num_frames = min(tssci_data.shape[0], int(duration * fps))
    frame_text = fig.text(0.5, 0.02, '', ha='center', fontsize=10,
                         bbox=dict(boxstyle='round', facecolor='wheat'))
    
    def animate(frame_idx):
        # Update scanning line
        tssci_row = int((frame_idx / num_frames) * 49)
        tssci_row = min(tssci_row, 48)
        scan_line.set_ydata([tssci_row, tssci_row])
        
        # Update skeleton
        frame_data = tssci_data[frame_idx]
        x_coords = frame_data[:, 0]
        y_coords = frame_data[:, 1]
        
        for i, (p1, p2) in enumerate(connections):
            if p1 < len(x_coords) and p2 < len(x_coords):
                lines[i].set_data([x_coords[p1], x_coords[p2]],
                                 [y_coords[p1], y_coords[p2]])
        
        points.set_data(x_coords, y_coords)
        frame_text.set_text(f'Frame: {frame_idx + 1}/{num_frames} | TSSCI Row: {tssci_row + 1}/49')
        
        return lines + [points, scan_line, frame_text]
    
    anim = animation.FuncAnimation(fig, animate, frames=num_frames,
                                  interval=1000/fps, blit=True)
    
    anim.save(output_path, writer='pillow', fps=fps)
    plt.close()
    print(f"✓ Dual-view animation saved to: {output_path}")

# Create dual-view animation
print("Creating dual-view animation (this may take a moment)...")
create_dual_view_animation(tssci_image, tssci_data, 
                          'tssci_output/dual_view_animation.gif',
                          fps=15, duration=5)

# Display the animation
display(IPImage('tssci_output/dual_view_animation.gif'))

## 8. Save All Results

Save all generated data and visualizations.

In [ ]:
# Save numpy arrays
np.save('tssci_output/skeleton_data.npy', skeleton_data)
np.save('tssci_output/tssci_data.npy', tssci_data)
np.save('tssci_output/tssci_image.npy', tssci_image)

# Save metadata
metadata = {
    'video_filename': video_filename,
    'video_fps': float(video_fps),
    'total_frames': int(skeleton_data.shape[0]),
    'skeleton_shape': list(skeleton_data.shape),
    'tssci_data_shape': list(tssci_data.shape),
    'tssci_image_shape': list(tssci_image.shape),
    'duration_seconds': float(skeleton_data.shape[0] / video_fps)
}

with open('tssci_output/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✓ All results saved to tssci_output/")
print("\nGenerated files:")
!ls -lh tssci_output/

# Display metadata
print("\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

## 9. Download Results

Download all generated files to your computer.

In [ ]:
# Compress output folder
!zip -r tssci_output.zip tssci_output/

# Download
print("Downloading results...")
files.download('tssci_output.zip')

print("\n✓ Download complete!")
print("\nYour zip file contains:")
print("  - skeleton_data.npy (raw MediaPipe skeleton)")
print("  - tssci_data.npy (TSSCI transformed skeleton)")
print("  - tssci_image.npy (TSSCI image for CNN)")
print("  - tssci_visualization.png (TSSCI channels)")
print("  - skeleton_animation.gif (skeleton animation)")
print("  - dual_view_animation.gif (TSSCI + skeleton)")
print("  - metadata.json (project information)")

## 10. Summary and Next Steps

### What We Accomplished

✅ **Skeleton Extraction:** Used MediaPipe to detect 33 body landmarks  
✅ **TSSCI Transformation:** Converted to 49-point TSSCI format  
✅ **Image Generation:** Created 49×49×2 TSSCI images  
✅ **Visualization:** Generated animations and dual-view display  

### Next Steps for Machine Learning

1. **Collect Dataset:**
   - Process multiple videos of different exercises
   - Generate TSSCI images for each
   - Label with exercise types

2. **Train EfficientNet-B7:**
   ```python
   # Load TSSCI images
   # Train CNN classifier
   # Evaluate on test set
   ```

3. **Deploy Model:**
   - Real-time exercise classification
   - Feedback for physiotherapy patients

### References

- **Research Paper:** [Segal, Y. (2023)](https://www.preprints.org/manuscript/202304.1268/v1)
- **MediaPipe:** [Google MediaPipe Pose](https://developers.google.com/mediapipe/solutions/vision/pose_landmarker)
- **Code Repository:** [GitHub - TSSCI-Pipeline](https://github.com/edialagerlov1/TSSCI-Pipeline)

---

**Congratulations! You've successfully created TSSCI images from your video!** 🎉